# ABLA — Controlled Ablation of Analysis-Window Duration

This notebook asks: **How much post-contact time is needed for multi-temperature LOMO effusivity prediction?**

It evaluates endpoints from 1.0 to 5.0 s at 0.5 s increments while holding constant:

- absolute smoothed sensor inputs (no baseline centering and no z-score scaling);
- summarized ESN features only;
- fixed ESN and XGBoost hyperparameters;
- material folds and reservoir seeds; and
- contact detection and target definition.

Because there are only nine discrete candidates, exhaustive evaluation is preferred over Bayesian optimization. The best observed endpoint is an **exploratory candidate**, not an unbiased performance estimate selected on an independent test set.

# 1. Imports and predeclared settings

In [ ]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import linalg
from scipy.signal import savgol_filter
from sklearn.compose import TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor


In [ ]:
TEMPERATURE_FOLDERS = ("30C", "40C", "50C", "60C")
TARGET = "eff"
WINDOW_ENDPOINTS = tuple(np.arange(1.0, 5.01, 0.5).round(2))
MAXIMUM_WINDOW = (0.0, max(WINDOW_ENDPOINTS))

ESN_PARAMS = {"res_size": 20, "leak_rate": 0.30, "input_magnitude": 1.00, "spectral_radius": 0.90, "washout": 0}
XGB_PARAMS = {
    "n_estimators": 300, "max_depth": 3, "learning_rate": 0.05,
    "subsample": 0.80, "colsample_bytree": 0.80, "min_child_weight": 5,
    "reg_alpha": 0.10, "reg_lambda": 10.0,
}
# One seed is faster for the complete duration scan. Use (42, 43, 44) for the final confirmatory run.
RESERVOIR_SEEDS = (42,)
RANDOM_STATE = 42
CLIP_PREDICTIONS_TO_TRAIN_RANGE = True

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, cwd.parent) if (p / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from the project root or notebooks directory.")
DATA_ROOT = PROJECT_ROOT / "data" / "02_preprocessed"
RESULTS_DIR = PROJECT_ROOT / "results" / "ablation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULT_STEM = "ABLA_analysis_window_scan_fixed_HP_absolute_smoothed_raw_summarized_ESN_only_LOMO"
print("Window endpoints:", WINDOW_ENDPOINTS)
print("ESN HPs:", ESN_PARAMS)
print("XGBoost HPs:", XGB_PARAMS)
print("Reservoir seeds:", RESERVOIR_SEEDS)


## Screening versus confirmation

The default scan uses seed 42 to keep nine-window evaluation practical. After identifying a stable region—not merely the single numerical winner—rerun the final two or three candidate endpoints with seeds 42, 43, and 44. Do not alter the fixed HPs.

# 2. Load data and compute standard effusivity targets

In [ ]:
STANDARD_PROPERTIES = {
    "ps_foam": {"k": .034, "rho": 25., "cp": 1400.}, "pu_foam": {"k": .043, "rho": 30., "cp": 1400.},
    "cork": {"k": .043, "rho": 240., "cp": 1800.}, "wood": {"k": .150, "rho": 700., "cp": 1700.},
    "pdms": {"k": .150, "rho": 970., "cp": 1460.}, "gypsum": {"k": .170, "rho": 800., "cp": 1090.},
    "cement": {"k": .290, "rho": 1440., "cp": 750.}, "graphite": {"k": 100., "rho": 1820., "cp": 710.},
    "bismuth": {"k": 8.1, "rho": 9780., "cp": 130.}, "titanium": {"k": 21.9, "rho": 4506., "cp": 523.},
    "nickel": {"k": 90.9, "rho": 8908., "cp": 461.}, "iron": {"k": 80.4, "rho": 7874., "cp": 449.},
    "aluminum": {"k": 237., "rho": 2700., "cp": 897.}, "copper": {"k": 401., "rho": 8960., "cp": 385.},
}
ALIASES = {
    "ps": "ps_foam", "ps foam": "ps_foam", "ps_foam": "ps_foam", "pu": "pu_foam", "pu foam": "pu_foam", "pu_foam": "pu_foam",
    "cork": "cork", "cork fine": "cork", "cork_fine": "cork", "wood": "wood", "pdms": "pdms", "gypsum": "gypsum",
    "cement": "cement", "graphite": "graphite", "carbon": "graphite", "bi": "bismuth", "bismuth": "bismuth",
    "ti": "titanium", "titanium": "titanium", "ni": "nickel", "nickel": "nickel", "fe": "iron", "iron": "iron",
    "al": "aluminum", "aluminum": "aluminum", "cu": "copper", "copper": "copper",
}
required = {"Sample", "Trial", "Time", "Primary", "Secondary"}
frames = []
for temperature in TEMPERATURE_FOLDERS:
    for path in sorted((DATA_ROOT / temperature).glob("*.csv")):
        frame = pd.read_csv(path)
        if required - set(frame.columns):
            continue
        frame["Temperature"] = temperature
        frames.append(frame)
if not frames: raise ValueError("No valid trial files found.")
DATA = pd.concat(frames, ignore_index=True).replace([np.inf, -np.inf], np.nan)
for column in ("Trial", "Time", "Primary", "Secondary"):
    DATA[column] = pd.to_numeric(DATA[column], errors="coerce")
DATA = DATA.dropna(subset=list(required)+["Temperature"]).copy()
DATA["Trial"] = DATA["Trial"].astype(int)
normalized = DATA["Sample"].astype(str).str.strip().str.lower().str.replace("_", " ")
DATA["Sample"] = normalized.map(ALIASES)
if DATA["Sample"].isna().any(): raise KeyError("Unknown material alias.")
for name in ("k", "rho", "cp"):
    DATA[name] = DATA["Sample"].map(lambda x: STANDARD_PROPERTIES[x][name])
DATA["eff"] = np.sqrt(DATA["k"]*DATA["rho"]*DATA["cp"])
DATA["trial_id"] = DATA["Temperature"] + "__" + DATA["Sample"] + "_trial_" + DATA["Trial"].astype(str)
DATA = DATA.sort_values(["trial_id", "Time"]).reset_index(drop=True)
print("Trials loaded:", DATA["trial_id"].nunique())


# 3. Detect contact once and retain the maximum 0–5 s interval

In [ ]:
def find_contact_time(trial, smooth_window=15, polyorder=2, threshold_frac=.30, skip_samples=5):
    clean = trial[["Time", "Primary"]].dropna().sort_values("Time").drop_duplicates("Time").reset_index(drop=True)
    time, signal = clean["Time"].to_numpy(float), clean["Primary"].to_numpy(float)
    if len(signal) < skip_samples+7 or np.any(np.diff(time)<=0): raise ValueError("Invalid sequence")
    time, signal = time[skip_samples:], signal[skip_samples:]
    window = min(smooth_window, len(signal)); window -= int(window%2==0)
    smooth = savgol_filter(signal, window, polyorder, mode="interp")
    derivative = np.gradient(smooth, time); strongest = int(np.argmin(derivative))
    active = derivative < threshold_frac*derivative[strongest]; elbow = 0
    for position in range(strongest, -1, -1):
        if not active[position]: elbow = position+1; break
    return float(time[elbow])

aligned_max = {}; rows = []
for trial_id, trial in DATA.groupby("trial_id", sort=False):
    trial = trial.sort_values("Time").drop_duplicates("Time").copy()
    try: contact = find_contact_time(trial)
    except ValueError as exc: warnings.warn(f"Skipping {trial_id}: {exc}"); continue
    trial["time_from_contact"] = trial["Time"]-contact
    trial = trial[trial["time_from_contact"].between(*MAXIMUM_WINDOW)].copy()
    if len(trial)<10: continue
    aligned_max[trial_id] = trial.reset_index(drop=True)
    rows.append({"trial_id": trial_id, "Temperature": trial["Temperature"].iloc[0], "Sample": trial["Sample"].iloc[0], "Trial": int(trial["Trial"].iloc[0]), "eff": float(trial["eff"].iloc[0])})
METADATA = pd.DataFrame(rows).sort_values("trial_id").reset_index(drop=True)
print("Aligned trials retained:", len(METADATA))

def windowed_trial(trial_id, endpoint):
    trial = aligned_max[trial_id]
    selected = trial[trial["time_from_contact"].between(0, endpoint)].copy()
    if len(selected)<5: raise ValueError(f"Too few samples for {trial_id}, endpoint={endpoint}")
    return selected.reset_index(drop=True)


# 4. Absolute smoothed ESN inputs and fixed reservoir

The ESN receives the smoothed absolute sensor values directly. No value at $t=0$ is subtracted, and no fold-local `StandardScaler` is applied.

In [ ]:
def smooth_signal(values, window=11, polyorder=2):
    values=np.asarray(values,float); selected=min(window,len(values)); selected-=int(selected%2==0)
    return savgol_filter(values,selected,polyorder,mode="interp") if selected>=5 else values.copy()

def esn_input(trial):
    time=trial["time_from_contact"].to_numpy(float)
    primary=smooth_signal(trial["Primary"]); secondary=smooth_signal(trial["Secondary"])
    return np.column_stack([primary,secondary,primary-secondary,np.gradient(primary,time),np.gradient(secondary,time)])

class ManualReservoir:
    def __init__(self, seed):
        rng=np.random.default_rng(seed); size=ESN_PARAMS["res_size"]
        self.Win=(rng.random((size,6))-.5)*ESN_PARAMS["input_magnitude"]
        W=rng.random((size,size))-.5; radius=np.max(np.abs(linalg.eigvals(W)))
        self.W=(W/radius.real)*ESN_PARAMS["spectral_radius"]; self.size=size
    def run(self, sequence):
        x=np.zeros((self.size,1)); states=[]; leak=ESN_PARAMS["leak_rate"]
        for row in np.asarray(sequence,float):
            proposed=np.tanh(self.Win@np.r_[1.,row].reshape(-1,1)+self.W@x)
            x=(1-leak)*x+leak*proposed; states.append(x[:,0].copy())
        return np.asarray(states)

def summarize_states(time, states):
    output={}
    for unit in range(states.shape[1]):
        x=states[:,unit]; prefix=f"esn_u{unit:03d}"
        output.update({
            f"{prefix}_mean":float(x.mean()), f"{prefix}_std":float(x.std()),
            f"{prefix}_min":float(x.min()), f"{prefix}_max":float(x.max()),
            f"{prefix}_initial":float(x[0]), f"{prefix}_net_change":float(x[-1]-x[0]),
            f"{prefix}_slope":float(np.polyfit(time,x,1)[0]), f"{prefix}_mean_abs":float(np.mean(np.abs(x))),
            f"{prefix}_abs_auc":float(np.trapezoid(np.abs(x),time)),
        })
    return output


# 5. Leakage-safe LOMO evaluator for one endpoint

In [ ]:
def make_model(seed):
    xgb=XGBRegressor(objective="reg:squarederror",random_state=int(seed),n_jobs=-1,tree_method="hist",verbosity=0,**XGB_PARAMS)
    base=Pipeline([("imputer",SimpleImputer(strategy="median")),("xgb",xgb)])
    return TransformedTargetRegressor(regressor=base,func=np.log1p,inverse_func=np.expm1,check_inverse=False)

def metrics(y_true,y_pred):
    y_true=np.asarray(y_true,float); y_pred=np.asarray(y_pred,float); rmse=float(np.sqrt(mean_squared_error(y_true,y_pred)))
    return {"n":len(y_true),"mae":float(mean_absolute_error(y_true,y_pred)),"rmse":rmse,"nrmse_range":rmse/float(np.ptp(y_true)),"r2":float(r2_score(y_true,y_pred)),"median_ape_pct":float(np.median(np.abs((y_true-y_pred)/y_true))*100)}

def build_features(endpoint,train_ids,all_ids,seed):
    # train_ids is retained for a consistent LOMO interface; absolute inputs require no fitted transform.
    reservoir=ManualReservoir(seed); rows=[]
    for trial_id in all_ids:
        trial=windowed_trial(trial_id,endpoint); time=trial["time_from_contact"].to_numpy(float)
        states=reservoir.run(esn_input(trial))
        row=summarize_states(time,states); row["trial_id"]=trial_id; rows.append(row)
    return pd.DataFrame(rows).set_index("trial_id")

def evaluate_endpoint(endpoint):
    predictions=[]; folds=[]; logo=LeaveOneGroupOut()
    for fold,(train_idx,test_idx) in enumerate(logo.split(METADATA,groups=METADATA["Sample"]),1):
        train_meta,test_meta=METADATA.iloc[train_idx],METADATA.iloc[test_idx]
        train_ids,test_ids=train_meta["trial_id"].tolist(),test_meta["trial_id"].tolist()
        y_train,y_test=train_meta[TARGET].to_numpy(float),test_meta[TARGET].to_numpy(float)
        seed_predictions=[]
        for seed in RESERVOIR_SEEDS:
            features=build_features(endpoint,train_ids,train_ids+test_ids,seed)
            model=make_model(seed); model.fit(features.loc[train_ids],y_train)
            pred=model.predict(features.loc[test_ids])
            if CLIP_PREDICTIONS_TO_TRAIN_RANGE: pred=np.clip(pred,y_train.min(),y_train.max())
            seed_predictions.append(pred)
        pred=np.mean(seed_predictions,axis=0); residual=pred-y_test; rmse=float(np.sqrt(np.mean(residual**2)))
        folds.append({"window_end_s":endpoint,"fold":fold,"held_out_material":test_meta["Sample"].iloc[0],"rmse":rmse,"nrmse_training_range":rmse/float(np.ptp(y_train)),"bias":float(residual.mean())})
        for i,(_,row) in enumerate(test_meta.iterrows()):
            predictions.append({"window_end_s":endpoint,"fold":fold,"trial_id":row["trial_id"],"Temperature":row["Temperature"],"Sample":row["Sample"],"Trial":row["Trial"],"y_true":y_test[i],"y_pred":pred[i]})
    predicted=pd.DataFrame(predictions); fold_table=pd.DataFrame(folds)
    overall={"window_end_s":endpoint,**metrics(predicted["y_true"],predicted["y_pred"]),"fold_nrmse_sd":float(fold_table["nrmse_training_range"].std(ddof=1))}
    overall["J_exploratory"]=overall["nrmse_range"]+.10*(1-overall["r2"])+.10*overall["fold_nrmse_sd"]
    return overall,fold_table,predicted


# 6. Exhaustively evaluate all candidate endpoints

In [ ]:
overall_rows=[]; fold_tables=[]; prediction_tables=[]
for position,endpoint in enumerate(WINDOW_ENDPOINTS,1):
    print(f"Window {position:02d}/{len(WINDOW_ENDPOINTS)}: 0–{endpoint:.1f} s")
    overall,folds,predicted=evaluate_endpoint(float(endpoint))
    overall_rows.append(overall); fold_tables.append(folds); prediction_tables.append(predicted)
WINDOW_RESULTS=pd.DataFrame(overall_rows).sort_values("window_end_s").reset_index(drop=True)
WINDOW_FOLDS=pd.concat(fold_tables,ignore_index=True); WINDOW_OOF=pd.concat(prediction_tables,ignore_index=True)
display(WINDOW_RESULTS)
exploratory_best=WINDOW_RESULTS.loc[WINDOW_RESULTS["J_exploratory"].idxmin()]
print(f"Exploratory best endpoint: {exploratory_best['window_end_s']:.1f} s; J={exploratory_best['J_exploratory']:.4f}")


# 7. Performance-versus-window and fold-stability analysis

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(17,5))
axes[0].plot(WINDOW_RESULTS["window_end_s"],WINDOW_RESULTS["nrmse_range"],"o-"); axes[0].set(title="Pooled LOMO NRMSE",xlabel="Window endpoint (s)",ylabel="NRMSE")
axes[1].plot(WINDOW_RESULTS["window_end_s"],WINDOW_RESULTS["r2"],"o-",color="#4daf4a"); axes[1].set(title="Pooled LOMO R²",xlabel="Window endpoint (s)",ylabel="R²")
axes[2].plot(WINDOW_RESULTS["window_end_s"],WINDOW_RESULTS["fold_nrmse_sd"],"o-",color="#984ea3"); axes[2].set(title="Fold-error variability",xlabel="Window endpoint (s)",ylabel="SD of fold NRMSE")
for ax in axes: ax.axvline(exploratory_best["window_end_s"],color="red",linestyle="--",alpha=.6); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()

heatmap=WINDOW_FOLDS.pivot(index="held_out_material",columns="window_end_s",values="nrmse_training_range")
fig,ax=plt.subplots(figsize=(12,7)); image=ax.imshow(heatmap,aspect="auto",cmap="viridis")
ax.set_xticks(range(len(heatmap.columns)),[f"{x:.1f}" for x in heatmap.columns]); ax.set_yticks(range(len(heatmap.index)),heatmap.index)
ax.set(xlabel="Window endpoint (s)",ylabel="Held-out material",title="Per-material LOMO error across analysis windows")
fig.colorbar(image,ax=ax,label="Training-range NRMSE"); plt.tight_layout(); plt.show()


# 8. Case-study prediction behavior across windows

In [ ]:
CASE_STUDY_TEMPERATURE="60C"
CASE_STUDY_MATERIALS=["ps_foam","wood","graphite","copper"]
case=(WINDOW_OOF[(WINDOW_OOF["Temperature"]==CASE_STUDY_TEMPERATURE)&WINDOW_OOF["Sample"].isin(CASE_STUDY_MATERIALS)]
      .groupby(["window_end_s","Sample"],as_index=False).agg(y_true=("y_true","mean"),y_pred=("y_pred","mean"),prediction_sd=("y_pred","std")))
fig,axes=plt.subplots(1,len(CASE_STUDY_MATERIALS),figsize=(19,4),sharex=True)
for ax,material in zip(axes,CASE_STUDY_MATERIALS):
    part=case[case["Sample"]==material].sort_values("window_end_s")
    ax.axhline(part["y_true"].iloc[0],color="black",label="Actual")
    ax.plot(part["window_end_s"],part["y_pred"],"o--",label="Mean OOF prediction")
    ax.fill_between(part["window_end_s"],part["y_pred"]-part["prediction_sd"],part["y_pred"]+part["prediction_sd"],alpha=.16)
    ax.set(yscale="log",title=material,xlabel="Window endpoint (s)")
axes[0].set_ylabel("Effusivity"); axes[0].legend(fontsize=8); fig.suptitle(f"Prediction sensitivity to window duration at {CASE_STUDY_TEMPERATURE}")
plt.tight_layout(); plt.show()


# 9. Detailed reservoir dynamics for one customizable case-study material

In [ ]:
DYNAMICS_MATERIAL="graphite"
DYNAMICS_TEMPERATURE="60C"
DYNAMICS_WINDOWS=(1.0,2.0,3.0,4.0,5.0)
DYNAMICS_SEED=42
DYNAMICS_UNITS=(0,3,7,11,15,19)

selected=METADATA[(METADATA["Sample"]==DYNAMICS_MATERIAL)&(METADATA["Temperature"]==DYNAMICS_TEMPERATURE)].sort_values("Trial")
if selected.empty: raise KeyError("Requested case-study material/temperature not found.")
fig,axes=plt.subplots(2,len(DYNAMICS_WINDOWS),figsize=(20,8),squeeze=False)
unit_colors=plt.cm.tab10(np.linspace(0,1,len(DYNAMICS_UNITS)))
for column,endpoint in enumerate(DYNAMICS_WINDOWS):
    common_time=np.linspace(0,endpoint,max(21,int(endpoint*20)+1)); sensor_trials=[]; state_trials=[]
    for trial_id in selected["trial_id"]:
        trial=windowed_trial(trial_id,endpoint); time=trial["time_from_contact"].to_numpy(float); represented=esn_input(trial)
        sensor_trials.append(np.column_stack([np.interp(common_time,time,represented[:,0]),np.interp(common_time,time,represented[:,1])]))
        states=ManualReservoir(DYNAMICS_SEED).run(represented)
        state_trials.append(np.column_stack([np.interp(common_time,time,states[:,unit]) for unit in DYNAMICS_UNITS]))
    sensor_trials=np.asarray(sensor_trials); state_trials=np.asarray(state_trials)
    sensor_mean,sensor_sd=sensor_trials.mean(0),sensor_trials.std(0,ddof=1); state_mean,state_sd=state_trials.mean(0),state_trials.std(0,ddof=1)
    for channel,(name,color) in enumerate((("Primary absolute","#1f77b4"),("Secondary absolute","#ff7f0e"))):
        axes[0,column].plot(common_time,sensor_mean[:,channel],color=color,label=name); axes[0,column].fill_between(common_time,sensor_mean[:,channel]-sensor_sd[:,channel],sensor_mean[:,channel]+sensor_sd[:,channel],color=color,alpha=.13)
    for index,unit in enumerate(DYNAMICS_UNITS):
        axes[1,column].plot(common_time,state_mean[:,index],color=unit_colors[index],label=f"Unit {unit}"); axes[1,column].fill_between(common_time,state_mean[:,index]-state_sd[:,index],state_mean[:,index]+state_sd[:,index],color=unit_colors[index],alpha=.10)
    axes[0,column].set_title(f"0–{endpoint:.1f} s"); axes[1,column].set_xlabel("Time (s)")
axes[0,0].set_ylabel("Absolute smoothed input"); axes[1,0].set_ylabel("Activation"); axes[0,-1].legend(fontsize=7); axes[1,-1].legend(fontsize=7,ncol=2)
fig.suptitle(f"Input and ESN dynamics across windows: {DYNAMICS_MATERIAL} at {DYNAMICS_TEMPERATURE}, repetition mean ± SD")
plt.tight_layout(); plt.show()


# 10. Should Bayesian optimization be used?

It can be implemented by allowing Optuna to nominate a continuous endpoint between 1 and 5 s. Nevertheless, this notebook deliberately uses a grid because:

1. only nine scientifically interpretable endpoints are being considered;
2. exhaustive evaluation is guaranteed to inspect every endpoint;
3. neighboring endpoints are strongly related prefixes of the same signal; and
4. Bayesian selection based directly on pooled outer LOMO predictions would reuse test-fold information and make the selected performance optimistic.

For an unbiased optimized-window estimate, the endpoint must be selected inside each outer LOMO training set using inner material-grouped validation. The selected endpoint may then differ among the 14 outer folds. That nested procedure answers a different question—performance of a window-selection algorithm—whereas the current ablation explains how performance changes with duration.

# 11. Save results and provenance

In [ ]:
WINDOW_RESULTS.to_csv(RESULTS_DIR/f"{RESULT_STEM}_overall_metrics.csv",index=False)
WINDOW_FOLDS.to_csv(RESULTS_DIR/f"{RESULT_STEM}_fold_metrics.csv",index=False)
WINDOW_OOF.to_csv(RESULTS_DIR/f"{RESULT_STEM}_oof_predictions.csv",index=False)
provenance={"window_endpoints":list(WINDOW_ENDPOINTS),"fixed_input_preprocessing":"absolute Savitzky-Golay-smoothed sensor values; no baseline centering, min-max scaling, or StandardScaler","esn_input_channels":["Primary absolute","Secondary absolute","Primary - Secondary","dPrimary/dt","dSecondary/dt"],"regressor_inputs":["summarized ESN trajectories only"],"esn_parameters":ESN_PARAMS,"xgb_parameters":XGB_PARAMS,"reservoir_seeds":list(RESERVOIR_SEEDS),"validation":"14-fold LOMO independently at every endpoint","selection_warning":"Best endpoint is exploratory unless confirmed independently or selected by nested CV."}
with (RESULTS_DIR/f"{RESULT_STEM}_provenance.json").open("w") as file: json.dump(provenance,file,indent=2)
print("Saved to",RESULTS_DIR)
